In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os

# ===================================
# ⚙️ Configuration
# ===================================
file_name = r"C:\Users\AZ\OneDrive\Desktop\LaligaFYP\MP2 DATASET.csv"

# ===================================
# 🧩 Load Dataset
# ===================================
try:
    df = pd.read_csv(file_name)

    # Clean up column names (fix trailing spaces)
    df.columns = df.columns.str.strip()

    if 'Year ' in df.columns:
        df.rename(columns={'Year ': 'Year'}, inplace=True)

    print("--- Data Loaded Successfully ---")
    print(df.head())

    print(f"\nDataset shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")

    # ===================================
    # 🔍 1. Missing Values Visualization
    # ===================================
    missing = df.isnull().sum().reset_index()
    missing.columns = ['Feature', 'Missing Count']
    missing = missing[missing['Missing Count'] > 0]

    if not missing.empty:
        fig_missing = px.bar(
            missing, x='Feature', y='Missing Count',
            title='Missing Values per Feature',
            text='Missing Count',
            color='Missing Count',
            color_continuous_scale='Reds'
        )
        fig_missing.show()
    else:
        print("✅ No missing values found.")

    # ===================================
    # 🧩 2. Duplicate Rows
    # ===================================
    duplicates = df.duplicated().sum()
    print(f"\nDuplicate rows: {duplicates}")

    fig_dup = go.Figure(go.Indicator(
        mode="number",
        value=duplicates,
        title={"text": "Number of Duplicate Rows"}
    ))
    fig_dup.show()

    # ===================================
    # 📊 3. Identify Column Types
    # ===================================
    numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

    print(f"\nNumerical Columns: {numerical_cols}")
    print(f"Categorical Columns: {categorical_cols}")

    # ===================================
    # 📈 4. Histograms (Numerical Columns)
    # ===================================
    print("\n--- Numerical Distributions ---")
    for col in numerical_cols:
        fig = px.histogram(df, x=col, title=f"Distribution of {col}",
                           marginal='box', color_discrete_sequence=['indianred'])
        fig.update_layout(bargap=0.1)
        fig.show()

    # ===================================
    # 📊 5. Bar Charts (Categorical Columns)
    # ===================================
    print("\n--- Categorical Distributions ---")
    for col in categorical_cols:
        value_counts = df[col].value_counts().nlargest(20)
        fig = px.bar(value_counts,
                     x=value_counts.index,
                     y=value_counts.values,
                     title=f'Distribution of {col} (Top 20)',
                     labels={'x': col, 'y': 'Count'},
                     color=value_counts.values,
                     color_continuous_scale='Blues')
        fig.show()

    # ===================================
    # 🔗 6. Scatter Plots (Feature Relationships)
    # ===================================
    print("\n--- Scatter Plot Relationships ---")
    if len(numerical_cols) >= 2:
        scatter_pairs = [(numerical_cols[i], numerical_cols[i+1]) for i in range(min(3, len(numerical_cols)-1))]
        for x_col, y_col in scatter_pairs:
            fig = px.scatter(df, x=x_col, y=y_col,
                             trendline='ols',
                             title=f"{y_col} vs {x_col}",
                             hover_data=['Team'] if 'Team' in df.columns else None)
            fig.show()
    else:
        print("Not enough numerical columns for scatter plots.")

    # ===================================
    # 📦 7. Box Plots (Outlier Detection)
    # ===================================
    print("\n--- Box Plots for Outliers ---")
    for col in numerical_cols:
        fig = px.box(df, y=col, title=f"Box Plot for {col}", points="outliers")
        fig.show()

    # ===================================
    # 🔥 8. Correlation Heatmap
    # ===================================
    print("\n--- Correlation Heatmap ---")
    if len(numerical_cols) >= 2:
        corr = df[numerical_cols].corr()
        fig = px.imshow(corr, text_auto=True, color_continuous_scale='Viridis',
                        title='Correlation Heatmap of Numerical Features')
        fig.show()
    else:
        print("Not enough numeric columns for correlation heatmap.")

    # ===================================
    # ⚖️ 9. Class Imbalance / Target Distribution
    # ===================================
    print("\n--- Checking for Class/Target Column ---")
    for target_col in ['class', 'Class', 'target', 'Target']:
        if target_col in df.columns:
            fig = px.pie(df, names=target_col, title=f'Class Distribution ({target_col})')
            fig.show()
            break
    else:
        print("No target column found for class imbalance check.")

    # ===================================
    # 📑 10. Summary Statistics
    # ===================================
    print("\n--- Summary Statistics ---")
    display(df.describe().T)

    print("\n✅ EDA Complete — All charts displayed interactively.")

except FileNotFoundError:
    print(f"❌ Error: The file '{file_name}' was not found. Please verify the path.")
except Exception as e:
    print(f"⚠️ An error occurred: {e}")


--- Data Loaded Successfully ---
   Rank         Team  Short Passes pg  Goals  Shots pg  Through Balls pg  \
0     1    Barcelona              560     98      15.6                 9   
1     2  Real Madrid              464    102      21.5                 8   
2     3     Valencia              402     59      14.3                 6   
3     4     Mallorca              264     57      12.8                 3   
4     5      Sevilla              341     66      13.2                 4   

   Aerials Won       Year  
0          8.3  2009-2010  
1          9.9  2009-2011  
2          9.2  2009-2012  
3         12.7  2009-2013  
4         12.3  2009-2014  

Dataset shape: (320, 8)
Columns: ['Rank', 'Team', 'Short Passes pg', 'Goals', 'Shots pg', 'Through Balls pg', 'Aerials Won', 'Year']
✅ No missing values found.

Duplicate rows: 0



Numerical Columns: ['Rank', 'Short Passes pg', 'Goals', 'Shots pg', 'Through Balls pg', 'Aerials Won']
Categorical Columns: ['Team', 'Year']

--- Numerical Distributions ---



--- Categorical Distributions ---



--- Scatter Plot Relationships ---



--- Box Plots for Outliers ---



--- Correlation Heatmap ---



--- Checking for Class/Target Column ---
No target column found for class imbalance check.

--- Summary Statistics ---


,count,mean,std,min,25%,50%,75%,max
Rank,320.0,10.500000,5.775312,1.0,5.750,10.50,15.250,20.0
Short Passes pg,320.0,364.068750,89.210618,212.0,302.000,345.50,403.000,694.0
Goals,320.0,50.806250,18.667020,22.0,38.000,46.00,58.000,120.0
Shots pg,320.0,12.181563,2.090328,7.1,10.775,11.90,13.000,21.5
Through Balls pg,320.0,1.968750,1.824328,0.0,1.000,1.00,2.000,13.0
Aerials Won,320.0,15.332187,3.910791,5.7,12.575,15.15,17.525,27.1



✅ EDA Complete — All charts displayed interactively.


In [2]:
%pip install plotly


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1 -> 25.3
[notice] To update, run: C:\Users\AZ\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os

# ===================================
# ⚙️ Configuration
# ===================================
file_name = r"C:\Users\AZ\OneDrive\Desktop\LaligaFYP\MP2 DATASET.csv"

# ===================================
# 🧩 Load Dataset
# ===================================
try:
    df = pd.read_csv(file_name)

    # Clean up column names (fix trailing spaces)
    df.columns = df.columns.str.strip()

    if 'Year ' in df.columns:
        df.rename(columns={'Year ': 'Year'}, inplace=True)

    print("--- Data Loaded Successfully ---")
    print(df.head())

    print(f"\nDataset shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")

    # ===================================
    # 🔍 1. Missing Values Visualization
    # ===================================
    missing = df.isnull().sum().reset_index()
    missing.columns = ['Feature', 'Missing Count']
    missing = missing[missing['Missing Count'] > 0]

    if not missing.empty:
        fig_missing = px.bar(
            missing, x='Feature', y='Missing Count',
            title='Missing Values per Feature',
            text='Missing Count',
            color='Missing Count',
            color_continuous_scale='Reds'
        )
        fig_missing.show()
    else:
        print("✅ No missing values found.")

    # ===================================
    # 🧩 2. Duplicate Rows
    # ===================================
    duplicates = df.duplicated().sum()
    print(f"\nDuplicate rows: {duplicates}")

    fig_dup = go.Figure(go.Indicator(
        mode="number",
        value=duplicates,
        title={"text": "Number of Duplicate Rows"}
    ))
    fig_dup.show()

    # ===================================
    # 📊 3. Identify Column Types
    # ===================================
    numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

    print(f"\nNumerical Columns: {numerical_cols}")
    print(f"Categorical Columns: {categorical_cols}")

    # ===================================
    # 📈 4. Histograms (Numerical Columns)
    # ===================================
    print("\n--- Numerical Distributions ---")
    for col in numerical_cols:
        fig = px.histogram(df, x=col, title=f"Distribution of {col}",
                           marginal='box', color_discrete_sequence=['indianred'])
        fig.update_layout(bargap=0.1)
        fig.show()

    # ===================================
    # 📊 5. Bar Charts (Categorical Columns)
    # ===================================
    print("\n--- Categorical Distributions ---")
    for col in categorical_cols:
        value_counts = df[col].value_counts().nlargest(20)
        fig = px.bar(value_counts,
                     x=value_counts.index,
                     y=value_counts.values,
                     title=f'Distribution of {col} (Top 20)',
                     labels={'x': col, 'y': 'Count'},
                     color=value_counts.values,
                     color_continuous_scale='Blues')
        fig.show()

    # ===================================
    # 🔗 6. Scatter Plots (Feature Relationships)
    # ===================================
    print("\n--- Scatter Plot Relationships ---")
    if len(numerical_cols) >= 2:
        scatter_pairs = [(numerical_cols[i], numerical_cols[i+1]) for i in range(min(3, len(numerical_cols)-1))]
        for x_col, y_col in scatter_pairs:
            fig = px.scatter(df, x=x_col, y=y_col,
                             trendline='ols',
                             title=f"{y_col} vs {x_col}",
                             hover_data=['Team'] if 'Team' in df.columns else None)
            fig.show()
    else:
        print("Not enough numerical columns for scatter plots.")

    # ===================================
    # 📦 7. Box Plots (Outlier Detection)
    # ===================================
    print("\n--- Box Plots for Outliers ---")
    for col in numerical_cols:
        fig = px.box(df, y=col, title=f"Box Plot for {col}", points="outliers")
        fig.show()

    # ===================================
    # 🔥 8. Correlation Heatmap
    # ===================================
    print("\n--- Correlation Heatmap ---")
    if len(numerical_cols) >= 2:
        corr = df[numerical_cols].corr()
        fig = px.imshow(corr, text_auto=True, color_continuous_scale='Viridis',
                        title='Correlation Heatmap of Numerical Features')
        fig.show()
    else:
        print("Not enough numeric columns for correlation heatmap.")

    # ===================================
    # ⚖️ 9. Class Imbalance / Target Distribution
    # ===================================
    print("\n--- Checking for Class/Target Column ---")
    for target_col in ['class', 'Class', 'target', 'Target']:
        if target_col in df.columns:
            fig = px.pie(df, names=target_col, title=f'Class Distribution ({target_col})')
            fig.show()
            break
    else:
        print("No target column found for class imbalance check.")

    # ===================================
    # 📑 10. Summary Statistics
    # ===================================
    print("\n--- Summary Statistics ---")
    display(df.describe().T)

    print("\n✅ EDA Complete — All charts displayed interactively.")

except FileNotFoundError:
    print(f"❌ Error: The file '{file_name}' was not found. Please verify the path.")
except Exception as e:
    print(f"⚠️ An error occurred: {e}")


--- Data Loaded Successfully ---
   Rank         Team  Short Passes pg  Goals  Shots pg  Through Balls pg  \
0     1    Barcelona              560     98      15.6                 9   
1     2  Real Madrid              464    102      21.5                 8   
2     3     Valencia              402     59      14.3                 6   
3     4     Mallorca              264     57      12.8                 3   
4     5      Sevilla              341     66      13.2                 4   

   Aerials Won       Year  
0          8.3  2009-2010  
1          9.9  2009-2011  
2          9.2  2009-2012  
3         12.7  2009-2013  
4         12.3  2009-2014  

Dataset shape: (320, 8)
Columns: ['Rank', 'Team', 'Short Passes pg', 'Goals', 'Shots pg', 'Through Balls pg', 'Aerials Won', 'Year']
✅ No missing values found.

Duplicate rows: 0



Numerical Columns: ['Rank', 'Short Passes pg', 'Goals', 'Shots pg', 'Through Balls pg', 'Aerials Won']
Categorical Columns: ['Team', 'Year']

--- Numerical Distributions ---



--- Categorical Distributions ---



--- Scatter Plot Relationships ---



--- Box Plots for Outliers ---



--- Correlation Heatmap ---



--- Checking for Class/Target Column ---
No target column found for class imbalance check.

--- Summary Statistics ---


,count,mean,std,min,25%,50%,75%,max
Rank,320.0,10.500000,5.775312,1.0,5.750,10.50,15.250,20.0
Short Passes pg,320.0,364.068750,89.210618,212.0,302.000,345.50,403.000,694.0
Goals,320.0,50.806250,18.667020,22.0,38.000,46.00,58.000,120.0
Shots pg,320.0,12.181563,2.090328,7.1,10.775,11.90,13.000,21.5
Through Balls pg,320.0,1.968750,1.824328,0.0,1.000,1.00,2.000,13.0
Aerials Won,320.0,15.332187,3.910791,5.7,12.575,15.15,17.525,27.1



✅ EDA Complete — All charts displayed interactively.


In [4]:
%pip install nbformat

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1 -> 25.3
[notice] To update, run: C:\Users\AZ\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os

# ===================================
# ⚙️ Configuration
# ===================================
file_name = r"C:\Users\AZ\OneDrive\Desktop\LaligaFYP\MP2 DATASET.csv"

# ===================================
# 🧩 Load Dataset
# ===================================
try:
    df = pd.read_csv(file_name)

    # Clean up column names (fix trailing spaces)
    df.columns = df.columns.str.strip()

    if 'Year ' in df.columns:
        df.rename(columns={'Year ': 'Year'}, inplace=True)

    print("--- Data Loaded Successfully ---")
    print(df.head())

    print(f"\nDataset shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")

    # ===================================
    # 🔍 1. Missing Values Visualization
    # ===================================
    missing = df.isnull().sum().reset_index()
    missing.columns = ['Feature', 'Missing Count']
    missing = missing[missing['Missing Count'] > 0]

    if not missing.empty:
        fig_missing = px.bar(
            missing, x='Feature', y='Missing Count',
            title='Missing Values per Feature',
            text='Missing Count',
            color='Missing Count',
            color_continuous_scale='Reds'
        )
        fig_missing.show()
    else:
        print("✅ No missing values found.")

    # ===================================
    # 🧩 2. Duplicate Rows
    # ===================================
    duplicates = df.duplicated().sum()
    print(f"\nDuplicate rows: {duplicates}")

    fig_dup = go.Figure(go.Indicator(
        mode="number",
        value=duplicates,
        title={"text": "Number of Duplicate Rows"}
    ))
    fig_dup.show()

    # ===================================
    # 📊 3. Identify Column Types
    # ===================================
    numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

    print(f"\nNumerical Columns: {numerical_cols}")
    print(f"Categorical Columns: {categorical_cols}")

    # ===================================
    # 📈 4. Histograms (Numerical Columns)
    # ===================================
    print("\n--- Numerical Distributions ---")
    for col in numerical_cols:
        fig = px.histogram(df, x=col, title=f"Distribution of {col}",
                           marginal='box', color_discrete_sequence=['indianred'])
        fig.update_layout(bargap=0.1)
        fig.show()

    # ===================================
    # 📊 5. Bar Charts (Categorical Columns)
    # ===================================
    print("\n--- Categorical Distributions ---")
    for col in categorical_cols:
        value_counts = df[col].value_counts().nlargest(20)
        fig = px.bar(value_counts,
                     x=value_counts.index,
                     y=value_counts.values,
                     title=f'Distribution of {col} (Top 20)',
                     labels={'x': col, 'y': 'Count'},
                     color=value_counts.values,
                     color_continuous_scale='Blues')
        fig.show()

    # ===================================
    # 🔗 6. Scatter Plots (Feature Relationships)
    # ===================================
    print("\n--- Scatter Plot Relationships ---")
    if len(numerical_cols) >= 2:
        scatter_pairs = [(numerical_cols[i], numerical_cols[i+1]) for i in range(min(3, len(numerical_cols)-1))]
        for x_col, y_col in scatter_pairs:
            fig = px.scatter(df, x=x_col, y=y_col,
                             trendline='ols',
                             title=f"{y_col} vs {x_col}",
                             hover_data=['Team'] if 'Team' in df.columns else None)
            fig.show()
    else:
        print("Not enough numerical columns for scatter plots.")

    # ===================================
    # 📦 7. Box Plots (Outlier Detection)
    # ===================================
    print("\n--- Box Plots for Outliers ---")
    for col in numerical_cols:
        fig = px.box(df, y=col, title=f"Box Plot for {col}", points="outliers")
        fig.show()

    # ===================================
    # 🔥 8. Correlation Heatmap
    # ===================================
    print("\n--- Correlation Heatmap ---")
    if len(numerical_cols) >= 2:
        corr = df[numerical_cols].corr()
        fig = px.imshow(corr, text_auto=True, color_continuous_scale='Viridis',
                        title='Correlation Heatmap of Numerical Features')
        fig.show()
    else:
        print("Not enough numeric columns for correlation heatmap.")

    # ===================================
    # ⚖️ 9. Class Imbalance / Target Distribution
    # ===================================
    print("\n--- Checking for Class/Target Column ---")
    for target_col in ['class', 'Class', 'target', 'Target']:
        if target_col in df.columns:
            fig = px.pie(df, names=target_col, title=f'Class Distribution ({target_col})')
            fig.show()
            break
    else:
        print("No target column found for class imbalance check.")

    # ===================================
    # 📑 10. Summary Statistics
    # ===================================
    print("\n--- Summary Statistics ---")
    display(df.describe().T)

    print("\n✅ EDA Complete — All charts displayed interactively.")

except FileNotFoundError:
    print(f"❌ Error: The file '{file_name}' was not found. Please verify the path.")
except Exception as e:
    print(f"⚠️ An error occurred: {e}")


--- Data Loaded Successfully ---
   Rank         Team  Short Passes pg  Goals  Shots pg  Through Balls pg  \
0     1    Barcelona              560     98      15.6                 9   
1     2  Real Madrid              464    102      21.5                 8   
2     3     Valencia              402     59      14.3                 6   
3     4     Mallorca              264     57      12.8                 3   
4     5      Sevilla              341     66      13.2                 4   

   Aerials Won       Year  
0          8.3  2009-2010  
1          9.9  2009-2011  
2          9.2  2009-2012  
3         12.7  2009-2013  
4         12.3  2009-2014  

Dataset shape: (320, 8)
Columns: ['Rank', 'Team', 'Short Passes pg', 'Goals', 'Shots pg', 'Through Balls pg', 'Aerials Won', 'Year']
✅ No missing values found.

Duplicate rows: 0



Numerical Columns: ['Rank', 'Short Passes pg', 'Goals', 'Shots pg', 'Through Balls pg', 'Aerials Won']
Categorical Columns: ['Team', 'Year']

--- Numerical Distributions ---



--- Categorical Distributions ---



--- Scatter Plot Relationships ---



--- Box Plots for Outliers ---



--- Correlation Heatmap ---



--- Checking for Class/Target Column ---
No target column found for class imbalance check.

--- Summary Statistics ---


,count,mean,std,min,25%,50%,75%,max
Rank,320.0,10.500000,5.775312,1.0,5.750,10.50,15.250,20.0
Short Passes pg,320.0,364.068750,89.210618,212.0,302.000,345.50,403.000,694.0
Goals,320.0,50.806250,18.667020,22.0,38.000,46.00,58.000,120.0
Shots pg,320.0,12.181563,2.090328,7.1,10.775,11.90,13.000,21.5
Through Balls pg,320.0,1.968750,1.824328,0.0,1.000,1.00,2.000,13.0
Aerials Won,320.0,15.332187,3.910791,5.7,12.575,15.15,17.525,27.1



✅ EDA Complete — All charts displayed interactively.


In [6]:
%pip install statsmodels


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1 -> 25.3
[notice] To update, run: C:\Users\AZ\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
